# Chapter 10 &mdash; Why Derivatives Matter: Negation and Intersection for Free

**Concept 2 of the Chapter 10 decomposition:** *Why Derivatives Matter: Negation and Intersection Without Determinization*

Troublesome operators are carried along symbolically instead of being expanded through determinization.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Negation-And-Intersection/Concept-Negation-And-Intersection.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Add **negation** ($!E$) and **intersection** ($E_1 \& E_2$) to regular expressions and
the classical route becomes painful: to complement you must determinize *first*, and
determinizing can be exponential.

With derivatives both are **one line each**:

$$(!E)_c = !(E_c) \qquad (E_1\&E_2)_c = (E_1)_c \ \&\ (E_2)_c$$

The operators are simply **carried along symbolically**. Nothing is expanded, nothing
is determinized. The work that determinization would have done is deferred to the
**nullability** test at the very end &mdash; where negation is just `not`.

That is why derivative matching is the technique of choice for RE dialects with
complement, and why it scales to grammars (Concept 8).

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### References built the classical way, for comparison

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

from itertools import product
STRS = [''.join(p) for k in range(9) for p in product('01', repeat=k)]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch10&nbsp;1.&nbsp;Brzozowski's Insight: an RE as its own State, Morphing as it Eats](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-RE-As-Its-Own-State/Concept-RE-As-Its-Own-State.ipynb) &nbsp;&middot;&nbsp; [**Chapter 10** index](https://github.com/ganeshutah/Jove/blob/master/Chapter10/README.md) &nbsp;&middot;&nbsp; [Ch10&nbsp;3.&nbsp;The Matching Algorithm: Steer by Derivatives, Decide by Nullability](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Matching-Algorithm/Concept-Matching-Algorithm.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Negation** costs one rule and no determinization.

In [ ]:
E = re2ast("!((0+1)*01)")[0]
D = comp_dfa(re_dfa("(0+1)*01"))
bad = [s for s in STRS if matches(s, E) != accepts_dfa(D, s)]
print("derivative negation vs comp_dfa : %d mismatches" % len(bad))
assert not bad
print("examples :", [s for s in STRS[:12] if matches(s, E)])

**Intersection** likewise.

In [ ]:
E = re2ast("((0+1)*01)&((0+1)*11)")[0]
D = intersect_dfa(re_dfa("(0+1)*01"), re_dfa("(0+1)*11"))
bad = [s for s in STRS if matches(s, E) != accepts_dfa(D, s)]
print("derivative intersection vs intersect_dfa : %d mismatches" % len(bad))
assert not bad
print("\n(nothing ends in both 01 and 11, so the language is empty:",
      any(matches(s, E) for s in STRS), ")")

A combination that the classical route makes genuinely awkward.

In [ ]:
r = "((0+1)*0(0+1)*) & !((0+1)*11(0+1)*)"
E = re2ast(r)[0]
spec = lambda s: ('0' in s) and ('11' not in s)
bad = [s for s in STRS if matches(s, E) != spec(s)]
print("contains a 0 but never 11 : %d mismatches" % len(bad))
assert not bad
print("accepted, short :", [s for s in STRS if matches(s, E)][:10])

The classical route needs determinization; the derivative route does not.

In [ ]:
D = intersect_dfa(re_dfa("(0+1)*0(0+1)*"), comp_dfa(re_dfa("(0+1)*11(0+1)*")))
print("classical: re2nfa -> nfa2dfa -> comp_dfa -> intersect_dfa -> min_dfa")
print("           final minimal machine has %d states" % len(min_dfa(D)["Q"]))
print("derivative: two extra lines in dv(), no machine at all")
assert all(matches(s, E) == accepts_dfa(D, s) for s in STRS)

## 4. Animation

What the classical route has to build; the derivative matcher builds none of it.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(intersect_dfa(re_dfa('(0+1)*0(0+1)*'), comp_dfa(re_dfa('(0+1)*11(0+1)*')))), FuseEdges=True)

## 5. Exercises


1. Add set difference $E_1 - E_2$ as a derivative rule. What is $(E_1-E_2)_c$?
2. Why does negation force determinization in the classical route?
3. Which of the two routes would you use inside a lexer generator? Why?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter10/Concept-Negation-And-Intersection')